## Import thư viện

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV

## Đọc file

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/DS108/Final_Project/Data/Gold_data.csv')

## Chia tập dữ liệu

In [ ]:
target = 'Price'
X = df.drop(target, axis=1)
y = df[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Kích thước tập huấn luyện:', X_train.shape)
print('Kích thước tập kiểm thử:', X_test.shape)


Kích thước tập huấn luyện: (193792, 153)
Kích thước tập kiểm thử: (48448, 153)


## Random Forest

### Mô hình với tham số mặc định và đánh giá bằng cross-validation

In [ ]:
model = RandomForestRegressor(n_estimators=100, random_state=42)

mae_train = -cross_val_score(model, x_train, y_train, cv=5, scoring='neg_mean_absolute_error')
r2_train = cross_val_score(model, x_train, y_train, cv=5, scoring='r2')

### Kết quả đánh giá bằng cross_validation

In [ ]:
print('Result on train data:\n')
print('   Mean MSE:', mae_train.mean())
print('   MSE standard deviation:', mae_train.std())
print('\n   Mean R2:', r2_train.mean())
print('   R2 standard deviation:', r2_train.std())
print('\n   R2 adjusted:', r2_adjusted(r2_train.mean(), x_train.shape[0], x_train.shape[1]))

Result on train data:

   Mean MSE: 104536.67026949309
   MSE standard deviation: 1158.6708959433352

   Mean R2: 0.9541919431826529
   R2 standard deviation: 0.0011643381683495606

   R2 adjusted: 0.9541557486717973


### Kết quả dự đoán trên tập test

In [ ]:
model.fit(x_train, y_train)
y_pred=model.predict(x_test)
print('Result on test data:\n')
print('R2 Score : ',r2_score(y_test,y_pred))
print('MAE : ', mean_absolute_error(y_test,y_pred) )
print('R2 adjusted:', r2_adjusted(r2_train.mean(), x_train.shape[0], x_train.shape[1]))

Result on test data:

R2 Score :  0.958693412654576
MAE :  95738.61030653336
R2 adjusted: 0.9541557486717973


### Tìm bộ siêu tham số tối ưu bằng gridSearch và kết quả

In [ ]:
param_grid = {
    'n_estimators': [50, 70, 100,150],
    'max_depth': [None,2, 5],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
}

# 4. Sử dụng GridSearchCV
grid_search = GridSearchCV(
    estimator=RandomForestRegressor(),
    param_grid=param_grid,
    cv=3,                # 3-fold cross validation
    scoring='r2',        # tối ưu R2
    n_jobs=4,           # sử dụng 4 cổng logic
    verbose=4
)

# 5. Fit mô hình
grid_search.fit(x_train, y_train)

# 6. In kết quả
print("Best parameters:", grid_search.best_params_)
print("Best score (R^2):", grid_search.best_score_)

# 7. Mô hình tốt nhất
best_model = grid_search.best_estimator_
y_pred=best_model.predict(x_test)
print('R2 Score : ',r2_score(y_test,y_pred))
print('MAE : ', mean_absolute_error(y_test,y_pred) )
print('R2 adjusted:', r2_adjusted(r2_train.mean(), x_train.shape[0], x_train.shape[1]))

Fitting 3 folds for each of 48 candidates, totalling 144 fits


/usr/local/lib/python3.11/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best parameters: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 150}
Best score (R^2): 0.949779970672694
R2 Score :  0.9588613352857006
MAE :  95551.89386596937
R2 adjusted: 0.9541557486717973
